# Few-Shot Learning

A refresher on **in-context learning**: teaching an LLM a task by *showing* it a handful of input→output examples inside the prompt — no fine-tuning, no weight updates, just demonstrations.

**Domain:** LLM Inference, Training & Optimization · **runnable:** yes

## 1. What & Why

**Few-shot learning** (a.k.a. **in-context learning**, ICL) means putting a few worked examples — "shots" — directly in the prompt so the model infers the task from the pattern and applies it to your real query. It's the headline result of GPT-3 ([Brown et al., 2020](https://arxiv.org/abs/2005.14165)): a large pretrained model can perform a brand-new task at inference time from a handful of demonstrations, with **zero gradient updates**.

**The problem it solves.** You have a task the model *can* do but won't do *the way you want* zero-shot: it picks the wrong label set, the wrong format, the wrong granularity, or it's just unreliable. Few-shot fixes this by demonstration instead of description — often the difference between a 60%- and a 95%-accurate prompt, in the time it takes to paste three examples.

**The spectrum:**

- **Zero-shot** — instruction only ("Classify the sentiment"). Cheapest; leans entirely on pretraining.
- **One-shot** — a single example to pin down format.
- **Few-shot** — typically **2–8** demonstrations that establish the label space, the output shape, and the decision boundary.

**When to reach for it:** when zero-shot gets the *idea* right but the *form* or *reliability* wrong, when you need a specific output format, or when the task has an unusual/closed label set the model wouldn't guess. **When not to:** when the model lacks the *knowledge* (use [[rag]]), when you need a fixed behavior at scale and prompts are getting huge (fine-tune — [[qlora]], [[finetune-transformer-lm]]), or when a clear instruction alone already works (don't pay the token tax for nothing).

Few-shot is one technique within the broader toolkit of [[prompt-engineering]]; this notebook drills into *how demonstrations work and how to choose them*.

## 2. Mental Model

**Show, don't tell.** A zero-shot prompt *describes* the task; a few-shot prompt *demonstrates* it and asks the model to continue the pattern. The model is a pattern-completion engine — give it three rows of `Input → Output` and a fourth `Input →`, and predicting the next tokens *is* doing the task.

A useful and surprisingly accurate frame: **the demonstrations are a tiny training set the model "learns" from in a single forward pass.** Recent theory ([Garg et al., 2022](https://arxiv.org/abs/2208.01066); [Xie et al., 2022](https://arxiv.org/abs/2111.02080)) shows transformers can implement learning algorithms (nearest-neighbor, regression) *over the in-context examples*. The weights never change — the "learning" is the model conditioning its output on the examples you supplied.

```
  ZERO-SHOT (describe)            FEW-SHOT (demonstrate)
  ┌───────────────────┐          ┌───────────────────────────────┐
  │ "Classify mood:"  │          │ "happy day"      -> positive   │  demonstrations
  │ "rainy monday" -> │          │ "lost my keys"   -> negative   │  (the in-context
  └─────────┬─────────┘          │ "great coffee"   -> positive   │   "training set")
            │                    │ "rainy monday"   ->            │  <- query, same shape
            ▼                    └──────────────┬────────────────┘
   model guesses format,                        ▼
   label set, tone                  model completes the pattern: "negative"
```

If the examples are good and *similar to the query*, the model has an easy analogy to follow. If they're irrelevant, contradictory, or skewed toward one label, the model follows *that* instead — which is exactly where few-shot goes wrong (Section 6).

## 3. Key Concepts

| Term | What it means |
|---|---|
| **In-context learning (ICL)** | Performing a task from prompt examples alone, with no weight updates. "Learning" happens in the forward pass. |
| **Shot / demonstration** | One `input → output` example in the prompt. *k*-shot = *k* of them. |
| **Demonstration selection** | *Which* examples to include. Static (fixed set) vs **dynamic/kNN** (retrieve examples most similar to the query — RAG over your example bank). The single biggest lever on quality. |
| **Demonstration ordering** | The *sequence* of examples. Models show **recency bias** (the last example weighs more) and order sensitivity — the same set in a different order can swing accuracy. |
| **Label space / format induction** | Demonstrations teach the *set of allowed outputs* and the *exact output shape* far more reliably than an instruction does. |
| **Majority-label & common-token bias** | The model drifts toward whatever label appears most among the shots, and toward tokens that are frequent/recent — a systematic bias you can **calibrate** out ([Zhao et al., 2021](https://arxiv.org/abs/2102.09690)). |
| **Format > correctness (the Min et al. finding)** | [Min et al., 2022](https://arxiv.org/abs/2202.12837) showed that *randomizing the labels* on demonstrations barely hurts: the model leans on the **format, label space, and input distribution** more than on the ground-truth pairings. Demonstrations teach *the shape of the task*, not just the answers. |
| **Token budget** | Every shot is tokens in *every* call — cost, latency, and context pressure scale with *k*. |

## 4. Setup

The mechanics of few-shot are about *prompt construction and example selection* — pure logic, no special libraries. The local examples below use only **NumPy + the standard library**, so they run in any fresh kernel on CPU in milliseconds.

The final example calls a real LLM (Anthropic's API) but is **gated behind an `os.getenv` check**, so the notebook executes top-to-bottom with or without a key. To run it for real:

```bash
%pip install anthropic
export ANTHROPIC_API_KEY=sk-ant-...
```

In [ ]:
import os
import numpy as np

rng = np.random.default_rng(0)  # fixed seed -> reproducible output
print("numpy", np.__version__)
print("ANTHROPIC_API_KEY set:", bool(os.getenv("ANTHROPIC_API_KEY")))

## 5. Worked Examples

### Example 1 — More shots = better, demonstrated honestly

To *show* in-context learning without a giant model, we model the LLM the way the theory says it behaves: as a learner that **only sees the in-context demonstrations** and classifies the query by analogy to them (a nearest-neighbor over the shots). Its "weights" never change — adding shots just gives it more context to condition on, exactly like real ICL.

The task is a synthetic 2-class rule the "model" has no prior knowledge of, so *all* the signal must come from the demonstrations. We sweep the number of shots and watch accuracy climb from chance toward solved.

In [ ]:
# Synthetic task: label = 1 if the point is inside a circle, else 0.
# The "model" has no built-in knowledge of this rule; it must infer it from shots.
def make_data(n):
    X = rng.uniform(-1, 1, size=(n, 2))
    y = (np.hypot(X[:, 0], X[:, 1]) < 0.6).astype(int)  # the hidden rule
    return X, y

pool_X, pool_y = make_data(400)      # bank we draw demonstrations from
test_X, test_y = make_data(300)      # held-out queries

def icl_predict(shots_X, shots_y, query_X):
    """In-context learner: classify each query by its nearest demonstration.
    With zero shots there is nothing to condition on -> fall back to a guess."""
    if len(shots_X) == 0:
        return np.zeros(len(query_X), dtype=int)  # no demos -> degenerate guess
    d = np.linalg.norm(query_X[:, None, :] - shots_X[None, :, :], axis=2)
    return shots_y[d.argmin(axis=1)]

print("shots | accuracy")
for k in [0, 1, 2, 4, 8, 16, 32]:
    idx = rng.choice(len(pool_X), size=k, replace=False) if k else []
    pred = icl_predict(pool_X[idx], pool_y[idx], test_X)
    acc = (pred == test_y).mean()
    print(f"{k:5d} | {acc:.3f}")

Zero-shot is stuck at the base rate (it can only guess); accuracy then climbs steeply with the first few shots and flattens. That diminishing-returns curve is the real-world signal that **2–8 well-chosen shots capture most of the gain** — piling on more mostly buys tokens, not accuracy.

### Example 2 — *Which* shots you pick beats *how many*

The highest-leverage move in production few-shot is **demonstration selection**: at a fixed budget of *k* shots, choosing examples *similar to the query* (dynamic / kNN few-shot) beats a random fixed set. Here we hold *k* constant and compare random selection against nearest-neighbor selection.

In [ ]:
def evaluate(selector, k, n_queries=300):
    """Accuracy when each query gets its own k demonstrations chosen by `selector`."""
    correct = 0
    for q in range(n_queries):
        idx = selector(test_X[q], k)
        pred = icl_predict(pool_X[idx], pool_y[idx], test_X[q:q+1])
        correct += int(pred[0] == test_y[q])
    return correct / n_queries

def random_selector(query, k):
    return rng.choice(len(pool_X), size=k, replace=False)

def knn_selector(query, k):
    d = np.linalg.norm(pool_X - query, axis=1)
    return d.argsort()[:k]  # the k demonstrations most similar to this query

k = 4
print(f"k={k} demonstrations per query")
print(f"  random selection : {evaluate(random_selector, k):.3f}")
print(f"  kNN  selection   : {evaluate(knn_selector, k):.3f}")

Same number of shots, same model — **relevant** demonstrations win by a wide margin. This is why production few-shot systems retrieve examples from a bank with an embedding index instead of hard-coding a fixed set: it's [[rag]]/[[semantic-search]], but retrieving *demonstrations* instead of *knowledge*.

A second, sneakier effect — **majority-label bias**: skew the label distribution of the shots and the model drifts toward the majority label, even when the query says otherwise.

In [ ]:
# Build skewed demo sets (mostly label 0) and see the prediction drift.
zeros = np.where(pool_y == 0)[0]
ones = np.where(pool_y == 1)[0]

def skewed_predict(n_zero, n_one):
    idx = np.concatenate([rng.choice(zeros, n_zero, replace=False),
                          rng.choice(ones,  n_one,  replace=False)])
    pred = icl_predict(pool_X[idx], pool_y[idx], test_X)
    return pred.mean()  # fraction predicted as class 1

print("demo balance (0s:1s) | fraction of queries predicted class 1")
for nz, no in [(6, 6), (10, 2), (2, 10)]:
    print(f"        {nz:2d}:{no:<2d}        | {skewed_predict(nz, no):.3f}")
print(f"\n(true fraction of class 1 in the test set: {test_y.mean():.3f})")

Balanced shots track the truth; lopsided shots pull predictions toward the majority class. The fix is to **balance the label distribution** of your demonstrations (and, for real LLMs, calibrate — see Section 6).

### Example 3 — The real thing (gated): a few-shot call to Claude

Production few-shot with the Messages API encodes each demonstration as an alternating **user → assistant** turn, then appends the real query as the final user turn. It runs **only if `ANTHROPIC_API_KEY` is set**, so the notebook still executes cleanly without one; the call shape is shown either way.

In [ ]:
DEMOS = [
    ("The new update bricked my device and support ghosted me.", "negative"),
    ("Setup took two minutes and it just works — love it.",        "positive"),
    ("It's a phone. Does phone things. No complaints, no thrills.", "neutral"),
]
QUERY = "Battery life is incredible, but the camera is mediocre."

def classify_with_claude(query, demos):
    import anthropic  # imported lazily so the notebook runs without the package
    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
    messages = []
    for text, label in demos:           # each shot = a user/assistant turn pair
        messages.append({"role": "user", "content": text})
        messages.append({"role": "assistant", "content": label})
    messages.append({"role": "user", "content": query})
    resp = client.messages.create(
        model="claude-opus-4-8",
        max_tokens=8,                   # we only want one word back
        system="You are a sentiment classifier. "
               "Reply with exactly one word: positive, negative, or neutral.",
        messages=messages,
    )
    return resp.content[0].text.strip().lower()

if os.getenv("ANTHROPIC_API_KEY"):
    print("Claude says:", classify_with_claude(QUERY, DEMOS))
else:
    print("ANTHROPIC_API_KEY not set — skipping the live call.")
    print(f"Would send model=claude-opus-4-8 with {len(DEMOS)} few-shot turns,")
    print(f"then the query: {QUERY!r}")

## 6. Gotchas & Pitfalls

- **Demonstration selection dominates.** *Which* examples you show matters more than *how many* (Example 2). Random or stale examples leave accuracy on the table; retrieve query-relevant shots when you can.
- **Majority-label bias.** Skew the labels of your shots and the model drifts toward the majority (Example 2). **Balance the label distribution** of demonstrations.
- **Recency & order sensitivity.** The last example carries extra weight, and the *same* set reordered can swing accuracy. Don't end on an outlier; if you can, evaluate a couple of orderings.
- **Common-token / surface bias — calibrate before use.** Models are biased toward labels that are frequent, recent, or first. [Zhao et al., 2021](https://arxiv.org/abs/2102.09690) ("Calibrate Before Use") fit a simple correction by probing with a content-free input (e.g. "N/A") and rescaling. Worth it for closed-set classification.
- **Format > correctness.** [Min et al., 2022](https://arxiv.org/abs/2202.12837) found randomizing the *labels* on shots barely hurts — the model leans on the *format and label space*. Takeaway: invest in getting the **shape, label set, and input distribution** of your demonstrations right; perfect ground-truth pairings matter less than you'd think.
- **More shots ≠ better.** Returns diminish fast (Example 1) while every shot costs tokens, latency, and context window on *every* call. Find the knee of the curve.
- **Demonstration leakage / overfitting to the bank.** If your shots are drawn from the same data you evaluate on, you'll overstate accuracy. Keep the example bank and the eval set disjoint.
- **Untrusted text in demonstrations.** Examples assembled from user/web data can carry prompt injections just like any other context — delimit and sanitize (see [[prompt-engineering]]).

## 7. When to Use vs Alternatives

| Approach | Best when | Cost / downside |
|---|---|---|
| **Zero-shot** | A clear instruction already works; the label set/format is obvious to the model. | Unreliable on unusual formats or closed label sets; nothing to anchor the output shape. |
| **Few-shot (ICL)** | Zero-shot has the right idea but wrong form/reliability; closed or unusual label set; you can supply 2–8 good examples. | Tokens on every call; sensitive to selection, order, and label balance. |
| **Dynamic / kNN few-shot** | You have a large bank of examples and queries vary — retrieve the most relevant shots per query. | Needs an embedding index + retrieval step (it's [[rag]] over examples). |
| **RAG** ([[rag]], [[semantic-search]]) | The model lacks *knowledge* (private/fresh facts), not just task framing. | Retrieval infrastructure; still need a good prompt to use the context. |
| **Fine-tuning** ([[qlora]], [[finetune-transformer-lm]]) | A fixed behavior/format at scale; prompts have grown huge/brittle; you want to bake the pattern into weights and cut per-call tokens. | Labeled data, training compute, a serving path; less flexible to change. |

**Rule of thumb:** zero-shot → few-shot → dynamic few-shot → fine-tune. Climb only when the cheaper rung genuinely can't do the job. Note the duality: **few-shot retrieves *demonstrations*, RAG retrieves *knowledge*, fine-tuning bakes *behavior* into weights** — and they compose (a fine-tuned model still benefits from good few-shot prompting).

## 8. Resources

- **Brown et al., 2020 — "Language Models are Few-Shot Learners" (GPT-3)**: https://arxiv.org/abs/2005.14165
- **Min et al., 2022 — "Rethinking the Role of Demonstrations"** (format > label correctness): https://arxiv.org/abs/2202.12837
- **Zhao et al., 2021 — "Calibrate Before Use"** (fixing majority/common-token bias): https://arxiv.org/abs/2102.09690
- **Liu et al., 2021 — "What Makes Good In-Context Examples for GPT-3?"** (kNN demonstration selection): https://arxiv.org/abs/2101.06804
- **Garg et al., 2022 — "What Can Transformers Learn In-Context?"** (ICL as learning algorithms): https://arxiv.org/abs/2208.01066
- **Anthropic — Multishot prompting guide**: https://platform.claude.com/docs/en/build-with-claude/prompt-engineering/multishot-prompting
- **DAIR.ai — Few-shot prompting**: https://www.promptingguide.ai/techniques/fewshot
- Related notebooks: [[prompt-engineering]], [[chain-of-thought]], [[rag]], [[semantic-search]].